In [1]:
!git clone https://github.com/fburlacu/czsl-prj.git


Cloning into 'czsl-prj'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 102 (delta 39), reused 97 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 204.18 KiB | 1.59 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [2]:
%cd czsl-prj

/content/czsl-prj


In [3]:
!pip install ftfy regex tqdm scipy pandas
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-mroz_hoh
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-mroz_hoh
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=899be0e244cb4d9d2f36a9f1922f7dc8abf4b1c1a78479190fea4a8e97e0a7cc
  Stored in directory: /tmp/pip-ephem-wheel-cache-f9p3qhr4/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [4]:
!cat CSP/download_data.sh

# Copyright (c) Facebook, Inc. and its affiliates.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.
#

CURRENT_DIR=$(pwd)

mkdir data
cd data

# download datasets and splits
wget -c http://wednesday.csail.mit.edu/joseph_result/state_and_transformation/release_dataset.zip -O mitstates.zip
wget -c http://vision.cs.utexas.edu/projects/finegrained/utzap50k/ut-zap50k-images.zip -O utzap.zip
wget -c https://senthilpurushwalkam.com/publications/compositional/compositional_split_natural.tar.gz -O compositional_split_natural.tar.gz
wget -c https://huggingface.co/datasets/nihalnayak/cgqa/resolve/main/cgqa.zip -O cgqa.zip


# MIT-States
unzip mitstates.zip 'release_dataset/images/*' -d mit-states/
mv mit-states/release_dataset/images mit-states/images/
rm -r mit-states/release_dataset
rename "s/ /_/g" mit-states/images/*

# UT-Zappos50k
unzip utzap.zip -d ut-zap50k/
mv ut-zap50k/ut-zap50k-images ut-zap

In [5]:
!sh CSP/download_data.sh

Streaming output truncated to the last 5000 lines.
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058615.382738.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058615.382739.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058621.382745.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058621.382746.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058621.382749.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058621.89020.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058672.382773.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058686.382775.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes/PUMA/8058696.151.jpg  
  inflating: ut-zap50k/ut-zap50k-images/Shoes/Sneaker

In [6]:
%cd data
!wget https://nlp.stanford.edu/data/glove.6B.zip


/content/czsl-prj/data
--2026-05-16 09:06:58--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-16 09:06:58--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.05MB/s    in 2m 39s  

2026-05-16 09:09:37 (5.17 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [8]:
import os

images_dir = "/content/czsl-prj/CSP/data/mit-states/images/"

for folder in os.listdir(images_dir):
    if " " in folder:
        old_path = os.path.join(images_dir, folder)
        new_path = os.path.join(images_dir, folder.replace(" ", "_"))
        os.rename(old_path, new_path)




In [ ]:
%cd /content/czsl-prj/CSP

!PYTHONPATH=/content/czsl-prj/GDE python -u train.py \
  --dataset mit-states \
  --clip_model ViT-L/14 \
  --experiment_name csp \
  --seed 0 \
  --epochs 20 \
  --lr 5e-05 \
  --attr_dropout 0.3 \
  --weight_decay 0.00001 \
  --train_batch_size 64 \
  --gradient_accumulation_steps 2 \
  --context_length 8 \
  --save_path /content/czsl-prj/CSP/data/model/mit-states/csp_model \
  --save_every_n 1

/content/czsl-prj/CSP
training details
Namespace(experiment_name='csp', dataset='mit-states', lr=5e-05, weight_decay=1e-05, clip_model='ViT-L/14', epochs=20, train_batch_size=64, eval_batch_size=1024, evaluate_only=False, context_length=8, attr_dropout=0.3, save_path='/content/czsl-prj/CSP/data/model/mit-states/csp_model', save_every_n=1, save_model=False, seed=0, gradient_accumulation_steps=2)
# train pairs: 1262 | # val pairs: 600 | # test pairs: 800
# train images: 30338 | # val images: 10420 | # test images: 12995
100%|███████████████████████████████████████| 890M/890M [00:15<00:00, 61.8MiB/s]
